# 03 · Datasets, DataLoaders, and batches
**Goals:** inspect a CSV, fit preprocessing only on training data, implement `Dataset`, and see how a `DataLoader` creates batches. The included synthetic data and its split are described in [data/README.md](../data/README.md).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

torch.manual_seed(486)
torch.set_num_threads(1)  # Small examples run efficiently on a laptop.
root = Path.cwd()
if not (root / "data" / "observations.csv").exists():
    root = root.parent
data_path = root / "data" / "observations.csv"
assert data_path.exists(), "Start Jupyter in the repository root or notebooks directory."
frame = pd.read_csv(data_path)
features = ["feature_1", "feature_2", "feature_3"]
train = frame[frame["split"] == "train"]
validation = frame[frame["split"] == "validation"]
test = frame[frame["split"] == "test"]
scaler = StandardScaler().fit(train[features])  # Never fit on validation or test.

def tensors(rows):
    x = torch.tensor(scaler.transform(rows[features]), dtype=torch.float32)
    y = torch.tensor(rows["label"].to_numpy(), dtype=torch.long)
    return x, y

x_train, y_train = tensors(train)
x_val, y_val = tensors(validation)
x_test, y_test = tensors(test)

print(frame.head())
print(frame.groupby("split")["label"].agg(["size", "mean"]))
print("training means used by scaler:", scaler.mean_)
print("transformed training feature means:", x_train.mean(dim=0))


`Dataset` describes an individual observation. Here the data fit in memory; for larger data, `__getitem__` can load an item on demand. The loader combines items into batches. `shuffle=True` reshuffles training order each epoch; validation/test order need not be shuffled.

In [ ]:
class TabularDataset(Dataset):
    def __init__(self, inputs, labels):
        self.inputs = inputs
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.inputs[index], self.labels[index]

training_data = TabularDataset(x_train, y_train)
loader = DataLoader(training_data, batch_size=64, shuffle=True, num_workers=0)
x_batch, y_batch = next(iter(loader))
print("one item:", training_data[0])
print("batch shapes:", x_batch.shape, y_batch.shape)
print("batch dtypes:", x_batch.dtype, y_batch.dtype)
print("batch sizes:", [len(y) for x, y in loader])


420 observations at batch size 64 give six full batches and a final batch of 36. The default `drop_last=False` retains it. For these classification labels use `long`; features use `float32`. `TensorDataset(x_train, y_train)` is a convenient built-in equivalent when data are already tensors.

## Try it
1. Change batch size to 100 and predict the number and sizes of batches.
2. Change `shuffle` and inspect several batches. Why is shuffled training useful?
3. Explain why fitting the scaler on the entire CSV leaks information.
4. Modify the Dataset to return the row index along with each example.